# 🧮 11 · Filtering and Histograms
### Convolution, kernels, and intensity transforms — the classical filtering toolbox

**Part B · Seeing like a microscope** · Module 11 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/11_filtering_and_histograms.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, SciPy, Matplotlib |
| **Prerequisites** | Modules 00–02 |
| **Estimated time** | 75–90 minutes |
| **02-740 topics covered** | 1-D/2-D convolution and kernels; mean and Gaussian smoothing; histograms and histogram equalization; intensity transforms; a first look at stain/colour unmixing |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every histogram and filtered image below is computed from a SIMULATED array built in this notebook; nothing is read from disk.

## 🎯 Learning objectives

1. Compute a 1-D and a 2-D convolution by hand, and explain the sliding-window algorithm underlying
   both.
2. Build and apply mean and Gaussian smoothing kernels, and explain the trade-off in choosing their
   size/sigma.
3. Compute and interpret an image's intensity histogram, and apply histogram equalization to improve
   contrast.
4. Explain, at a conceptual level, how colour/stain unmixing is a point-operation application built
   on the same linear-algebra ideas.

## 🗺️ Contents

1. 1-D convolution, by hand
2. 2-D convolution and kernels
3. Smoothing filters: mean vs. Gaussian, and the noise/detail trade-off
4. Histograms and histogram equalization
5. A glimpse ahead: colour/stain unmixing as a point operation

## 🧫 The biological question

Every synthetic fluorescence image you have simulated so far already has noise and blur baked in
(Module 02's point-spread function). Before any detection, segmentation, or classification
algorithm can run reliably, the image usually needs to be cleaned up — noise suppressed, contrast
improved — without destroying the real biological structures you actually care about. Filtering is
how that cleanup happens, and it is also, perhaps surprisingly, one of the operations a microscope's
own optics already perform on every image it forms (the point-spread function *is* a convolution).
Understanding convolution and filtering from first principles therefore serves two purposes at
once: it explains how real microscope images already got degraded, and it hands you the tools to
partially undo that degradation.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 3
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · 1-D convolution, by hand

For a 1-D signal $f$ and a kernel $\omega$ of half-width $a$, convolution is defined as

$$g(x) = (\omega * f)(x) = \sum_{s=-a}^{a} \omega(s)\, f(x - s)$$

*In words:* to compute the output at position $x$, flip the kernel, slide it so it is centered at
$x$, multiply it pointwise against the signal underneath it, and sum everything up. This single
operation — slide, multiply, sum — is the computational core of nearly every classical image-filtering
method in this course.

In [ ]:
# ── 1.1 a tiny 1-D convolution computed two ways: a hand-written loop, and np.convolve ──
signal = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 4.0, 3.0, 2.0, 1.0])
kernel = np.array([0.25, 0.5, 0.25])   # a simple 3-tap smoothing kernel

def convolve_1d_by_hand(f, w):
    a = len(w) // 2
    f_padded = np.pad(f, a, mode='edge')
    out = np.zeros_like(f, dtype=float)
    for x in range(len(f)):
        # flip the kernel (w[::-1]) and take the dot product with the local window
        window = f_padded[x:x + len(w)]
        out[x] = np.dot(w[::-1], window)
    return out

by_hand = convolve_1d_by_hand(signal, kernel)
by_numpy = np.convolve(signal, kernel, mode='same')
print('by hand :', np.round(by_hand, 3))
print('np.convolve:', np.round(by_numpy, 3))
print('match:', np.allclose(by_hand, by_numpy, atol=1e-6))

🔍 **What just happened?** Our hand-written sliding-window-dot-product loop produces exactly the
same result as `np.convolve`. This confirms the "slide, multiply (with the kernel flipped), sum"
recipe really is what convolution means — the built-in function is simply a much faster
implementation of the identical idea.

✏️ **Your turn.** Change `kernel` to `np.array([-1.0, 0.0, 1.0])` (a simple derivative-like
kernel) and re-run. Which parts of `signal` produce the largest-magnitude outputs? Relate this to
where `signal` changes most rapidly.

---
## 2 · 2-D convolution and kernels

2-D convolution is the direct extension of the 1-D idea: for an image $f$ and a kernel $\omega$
of half-size $(a, b)$,

$$g(x, y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} \omega(s, t)\, f(x - s, y - t)$$

A kernel is almost always a small, odd-sized square matrix (3×3, 5×5, ...). Centering the kernel on
every pixel in turn, multiplying element-wise, and summing, produces one output image the same size
as the input — this is exactly what a convolutional neural network's convolution layer does too
(Module 16), just with a *learned* kernel instead of a hand-designed one.

In [ ]:
# ── 2.1 a hand-designed 3x3 kernel applied with scipy.ndimage, compared against the raw image ──
from scipy import ndimage as ndi_demo

sim = simulate_nuclei_image(n_cells=12, size=100, seed=SEED)
image = sim['image']

sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float)
box_blur_kernel = np.ones((5, 5)) / 25.0

sharpened = ndi_demo.convolve(image, sharpen_kernel, mode='reflect')
box_blurred = ndi_demo.convolve(image, box_blur_kernel, mode='reflect')

show_image_grid([image, sharpened, box_blurred],
                 ['original (simulated)', 'sharpening kernel', '5x5 box-blur kernel'], ncols=3)
plt.show()

🔍 **What just happened?** The same `convolve` operation, with two different 3×3/5×5 kernels,
produces completely different effects: the sharpening kernel accentuates intensity differences
(making edges and noise both more prominent), while the box-blur kernel averages neighboring pixels
together, suppressing both noise and fine detail. The kernel's *values*, not the operation itself,
determine what a convolution does.

### 📘 Lecture notes — convolution and kernels

**Definition.** A convolution kernel (also called a filter or mask) is a small matrix that, when
slid across every position of an image and combined via the sum-of-products defined above,
transforms the image. The operation is linear and shift-invariant: the same kernel applied at any
location does the same thing, and the output is a weighted sum of the input, never a product of
whole matrices.

**Worked example.** A 3×3 mask centered at pixel $(x, y)$ reads 9 neighboring pixel values, each
multiplied by the mask's corresponding (flipped) entry, and the 9 products are summed to produce the
single output value at $(x, y)$ — repeated independently at every pixel in the image.

**One memorable point.** *Convolution is not matrix multiplication — the kernel and the local image
patch have matching shapes and are combined element-wise, then summed to a single scalar, not
matrix-multiplied in the linear-algebra sense.*

---
## 3 · Smoothing filters: mean vs. Gaussian, and the noise/detail trade-off

In [ ]:
# ── 3.1 noisy image, smoothed with mean filters and Gaussian filters at several scales ──
clean = sim['image']
noisy = add_gaussian_noise(clean, 0.08, rng)

sigmas = [1, 2, 5]
fig, axes = plt.subplots(2, len(sigmas) + 1, figsize=(12, 6))
show_image(axes[0, 0], clean, 'clean (ground truth)')
show_image(axes[1, 0], noisy, 'noisy input')
for j, s in enumerate(sigmas, start=1):
    mean_filtered = ndi.uniform_filter(noisy, size=int(4 * s + 1))
    gauss_filtered = blur_image(noisy, gaussian_psf(s))
    show_image(axes[0, j], mean_filtered, f'mean filter (size~{int(4*s+1)})')
    show_image(axes[1, j], gauss_filtered, f'Gaussian filter (sigma={s})')
plt.tight_layout(); plt.show()

🔍 **What just happened?** At small scale (sigma=1), both filters only mildly suppress noise and
preserve detail; at large scale (sigma=5) both filters substantially suppress noise, but also start
blurring away the nuclei's real edges and merging nearby nuclei together. This is the fundamental
smoothing trade-off: more averaging means more noise suppression, but also more loss of real
structural detail, so there is no free lunch — only a choice of where on this spectrum a given
analysis needs to sit.

✏️ **Your turn.** For the `noisy` image above, use `course_toolkit`'s `add_gaussian_noise` with a
much higher `sigma` (e.g. `0.25`) to make a very noisy version, then find, by trial and error, the
smallest Gaussian `sigma` (passed to `gaussian_psf`) that visually suppresses most of the noise
without obviously merging adjacent nuclei. There is no single correct answer — this is exactly the
kind of judgment call real image-analysis pipelines require.

🧪 **Ground-truth check.** Compute the mean squared error between each filtered image and the
`clean` ground truth (which you would never have for a real microscope image, but which you do have
here because this is a simulation): `np.mean((filtered - clean) ** 2)`. Does the sigma that
*looks* best by eye also have close to the lowest numerical error against ground truth? This
illustrates why simulated data with known ground truth (used throughout this course) is such a
powerful tool for objectively evaluating an algorithm, rather than relying on visual judgment
alone.

---
## 4 · Histograms and histogram equalization

An image's **histogram** counts how many pixels take each intensity value (or fall in each small
intensity bin) — the $x$-axis is intensity, the $y$-axis is the probability (or count) of that
intensity occurring. A low-contrast image (e.g. a faint fluorescence signal) often has a histogram
tightly bunched in a narrow intensity range. **Histogram equalization** is a point operation — it
remaps each intensity value through a function computed from the image's own cumulative histogram,
designed so that the *output* histogram is as close to flat (uniform) as possible, stretching out
the contrast of the most common intensity ranges.

In [ ]:
# ── 4.1 a deliberately low-contrast image, and its histogram before/after equalization ──
low_contrast = 0.3 + 0.15 * sim['image'] / (sim['image'].max() + 1e-9)  # compress into a narrow range

def histogram_equalize(image, n_bins=256):
    flat = image.ravel()
    hist, bin_edges = np.histogram(flat, bins=n_bins, range=(flat.min(), flat.max()))
    cdf = np.cumsum(hist).astype(float)
    cdf /= cdf[-1]                                   # normalize to [0, 1]
    equalized_flat = np.interp(flat, bin_edges[:-1], cdf)
    return equalized_flat.reshape(image.shape)

equalized = histogram_equalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
show_image(axes[0, 0], low_contrast, 'low-contrast image')
axes[0, 1].hist(low_contrast.ravel(), bins=64, color='gray'); axes[0, 1].set_title('its histogram')
show_image(axes[1, 0], equalized, 'after histogram equalization')
axes[1, 1].hist(equalized.ravel(), bins=64, color='gray'); axes[1, 1].set_title('equalized histogram')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The low-contrast image's histogram is a tall, narrow spike — almost
all pixels share nearly the same intensity, so structure is hard to see. After equalization, the
histogram is spread much more evenly across the full intensity range, and the nuclei become far
easier to distinguish from the background by eye — even though **no new information was added**;
equalization only *redistributes* the existing information for better visibility (and, often, for
better behavior of downstream thresholding algorithms in Module 18).

### 📘 Lecture notes — histograms and point processing

**Definition.** A point operation (or point processing / intensity transform) computes each output
pixel's value from *only* the corresponding input pixel's value (via some mapping function $T$),
with no dependence on neighboring pixels — unlike convolution, which is a neighborhood operation.
Histogram equalization is a point operation whose mapping function is derived from the image's own
cumulative intensity histogram.

**Worked example.** Applying histogram equalization to a clinical MRI image with subtle, hard-to-see
intensity differences near a fracture can make that fracture substantially easier for a radiologist
(or an algorithm) to detect, purely by redistributing existing contrast.

**One memorable point.** *Point operations (histogram equalization, thresholding, gamma correction)
and neighborhood operations (convolution, filtering) are the two basic building blocks of classical
image processing — nearly everything else is built by combining the two.*

---
## 5 · A glimpse ahead: colour/stain unmixing as a point operation

One more classical point-operation application worth knowing about, common in histology (staining
tissue sections with dyes like haematoxylin and eosin, or H&E, for pathology): a brightfield image
of stained tissue is really a *linear mixture* of the absorption spectra of each individual stain.
**Colour deconvolution** (or stain unmixing) inverts this known linear mixing model to recover an
estimated concentration image for each individual stain, given the combined, observed RGB image and
a reference matrix of each stain's known absorption characteristics. This is conceptually a direct
descendant of the same linear-algebra idea you will see formalized for dimensionality reduction in
Module 17 (and is a point operation applied per-pixel, combining all channels, rather than per
single-channel intensity) — think of it as "unmixing paint colours" rather than "mixing paint
colours", done one pixel at a time.

In [ ]:
# ── 5.1 a toy two-stain unmixing example (not a shipped toolkit function — small enough to inline) ──
# Simulate two "stain" concentration maps, then their known linear mixture into an observed 2-channel image.
stain_a_conc = np.clip(sim['image'], 0, None)                       # "stain A" follows the nuclei
stain_b_conc = np.clip(0.4 - 0.3 * sim['image'] / (sim['image'].max() + 1e-9), 0, None)  # "stain B" is complementary

mixing_matrix = np.array([[0.8, 0.3], [0.25, 0.9]])   # how much each stain contributes to each observed channel
observed = np.stack([
    mixing_matrix[0, 0] * stain_a_conc + mixing_matrix[0, 1] * stain_b_conc,
    mixing_matrix[1, 0] * stain_a_conc + mixing_matrix[1, 1] * stain_b_conc,
], axis=-1)

unmixing_matrix = np.linalg.inv(mixing_matrix)   # the whole trick: invert the KNOWN mixing matrix
h, w, _ = observed.shape
recovered = (observed.reshape(-1, 2) @ unmixing_matrix.T).reshape(h, w, 2)

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
show_image(axes[0], stain_a_conc, 'true stain A')
show_image(axes[1], stain_b_conc, 'true stain B')
show_image(axes[2], recovered[..., 0], 'recovered stain A')
show_image(axes[3], recovered[..., 1], 'recovered stain B')
plt.tight_layout(); plt.show()
print('max recovery error, stain A:', np.abs(recovered[..., 0] - stain_a_conc).max())

🧪 **Ground-truth check.** Because we built this simulation ourselves, we know the exact true
concentration maps for both stains — and, with a known, invertible mixing matrix and no noise, the
recovered concentrations match the true ones almost exactly (the tiny residual error is pure
floating-point roundoff). In a real experiment the mixing matrix must itself be estimated from
reference single-stain slides, and noise prevents a perfect recovery — but the core idea, inverting a
known or estimated linear mixing model, is exactly this.

## 📌 Key takeaways

- Convolution is a "slide, multiply, sum" sliding-window operation; the kernel's values, not the mechanism, determine the effect (smoothing, sharpening, and much more).
- Mean and Gaussian smoothing both trade noise suppression against loss of real detail; there is no universally correct amount of smoothing.
- A histogram summarizes an image's intensity distribution; histogram equalization is a point operation that redistributes existing contrast without adding new information.
- Colour/stain unmixing applies the same linear-algebra idea (inverting a known mixing model) to recover individual stain concentrations from a combined observed image.

## ✅ Checkpoint

**1.** In the convolution formula $g(x) = \sum_s \omega(s) f(x-s)$, why is the kernel described as
"flipped" relative to a simple local weighted average?

<details><summary>Answer</summary>
Because the kernel index $s$ is subtracted from $x$ rather than added, which reverses the kernel's
order relative to how it is normally written out and displayed; for symmetric kernels (like a
Gaussian) this flip has no visible effect, but for an asymmetric kernel (like a derivative kernel)
it matters.
</details>

**2.** Why might a very large Gaussian smoothing sigma hurt a downstream nucleus-counting
algorithm, even though it removes the most noise?

<details><summary>Answer</summary>
Because it can blur separate, nearby nuclei together into what looks like a single blob, causing an
undercount — too much smoothing destroys real structure, not just noise.
</details>

**3.** Does histogram equalization add new information to an image?

<details><summary>Answer</summary>
No — it only redistributes the existing intensity information to make it more visually
distinguishable (and often easier to threshold); no new measurement or detail is created.
</details>

## 🧠 Questions & discussion

- If smoothing always trades detail for noise suppression, can you think of any smoothing-free way
  to reduce noise instead? (This is a direct preview of what Module 24 covers in depth.)
- Histogram equalization can sometimes *overemphasize* noise in otherwise near-uniform background
  regions. Why might that happen, and what would you want to check before trusting an equalized
  image for quantitative measurement (rather than just for visualization)?
- Stain unmixing assumed a perfectly known, noise-free linear mixing matrix. What do you expect
  would happen to the recovered concentrations if the assumed mixing matrix were slightly wrong?

## 🚀 Bonus — going further

- Implement a *median* filter (replace each pixel with the median of its neighborhood, not the
  mean) using `scipy.ndimage.median_filter`, and compare its behavior on an image with
  salt-and-pepper noise (`add_salt_pepper_noise` is in the shared toolkit, though not copied into
  this notebook — look at its docstring in `course_toolkit.py`) versus Gaussian smoothing. Which
  handles impulse noise better, and why?
- Try CLAHE (Contrast-Limited Adaptive Histogram Equalization, in `skimage.exposure`) instead of
  global histogram equalization, and compare the results on an image with both very dark and very
  bright regions.
- Read about gamma correction ($g = f^{\gamma}$) as another common point-operation intensity
  transform, and compute its effect on `low_contrast` for a few different gamma values.

## 📝 Lab note

```text
Module: 11 — Filtering and Histograms
Date:
A smoothing sigma that worked well for my test image, and why:
One point operation I want to try on real data:
```

## 📚 References

1. Gonzalez, R.C., Woods, R.E. (2018). *Digital Image Processing* (4th ed.). Pearson.
2. Russ, J.C. (2015). *The Image Processing Handbook* (7th ed.). CRC Press.
3. Pratt, W.K. (2007). *Digital Image Processing* (4th ed.). Wiley.
4. Ruifrok, A.C., Johnston, D.A. (2001). Quantification of histochemical staining by color
   deconvolution. *Analytical and Quantitative Cytology and Histology*, 23(4), 291–299.
5. Zuiderveld, K. (1994). Contrast limited adaptive histogram equalization. In *Graphics Gems IV*,
   474–485.

## ➡️ Next

Module 12 switches from the spatial domain to the frequency domain: the 2-D Fourier transform,
low-pass/high-pass filtering, and multiscale (pyramid) analysis — a different, complementary lens on
exactly the same filtering ideas introduced here.